# 17-bridge CPW–meander · Driven Terminal / Lumped Ports

Excite the launcher variant through its two neutral nonmetal sheets.
Read the [Driven Terminal
requirement](../../../simulation/requirements/driven-terminal.qmd) for
the system and expected output. The same source supports can receive
[Eigenmode RLC treatments](aedt_hfss_eigenmode_rlc.ipynb) without
redrawing them.

Target: [SCGSim v2
RC1](https://github.com/OrPenStrike/scgsim/tree/85d8bedb733b1e17737d935029618351b30ea5de),
PDK candidate, AEDT 2024.2 / PyAEDT 1.3.0 in a separate selected
runtime. Source-only publication; no old native output is attributed to
this notebook.

## Action and identity

In [ ]:
import json
import subprocess
from importlib.metadata import version
from pathlib import Path
from uuid import uuid4

from IPython.display import display
from scgsim.aedt import (
    AedtResources,
    FrequencySweepSpec,
    HfssRunControl,
    LumpedTerminalPort,
    prepare_hfss_driven_from_geometry,
    prepare_planar_geometry_input,
    resolve_results,
)
from scgsim.geometry import (
    GeometryBuildInput,
    LayoutPolygonSpec,
    LumpedSupport,
    SemanticEntitySpec,
    VacuumRegionSpec,
    apply_vacuum_region_to_stack,
)

import orpen_sc_pdk
from orpen_sc_pdk.cells import cpw_meander_airbridge_lumped_coupon
from orpen_sc_pdk.cells.simulation import get_cpw_meander_airbridge_source_records

orpen_sc_pdk.activate()
WORKFLOW_ACTION = "author"  # author | prepare_handoff | run | analyze
VARIANT = "lumped"
RUN_ID = f"meander-terminal-lumped-{uuid4().hex[:12]}"
OUTPUT_ROOT = Path("notebooks/.artifacts/ComponentSimulation/CpwMeanderAirbridge")
RUN_DIR = OUTPUT_ROOT / RUN_ID
RETURNED_RUN_DIR = OUTPUT_ROOT / "returned-run"
ANALYSIS_KIND = "strict"  # strict | receipt_only
display({name: version(name) for name in ("scgsim", "gdsfactory", "orpen-sc-pdk")})

## Layout, stack and neutral sheets

<figure>
<img
src="https://raw.githubusercontent.com/OrPenStrike/orpen-sc-pdk/f4f509843c2c7177773798bf6a8a906f94c63a11/docs/_static/images/test-components/driven-terminal-lumped-resonator.png"
alt="Selected launcher source geometry; not native port readback." />
<figcaption aria-hidden="true">Selected launcher source geometry; not
native port readback.</figcaption>
</figure>

In [ ]:
if WORKFLOW_ACTION != "analyze":
    coupon = cpw_meander_airbridge_lumped_coupon()
    coupon.plot()

M1 Al is 100 nm; posts start at its upper face, with deck underside 3 µm
above that face and deck thickness 0.3 µm. Each launcher support is the
actual 202/1 nonmetal sheet at Z=0.05 µm, normal +Z, local dimensions
85×150 µm. Its ordered contacts run Signal→positive Ground. It is not a
conductor or a junction.

## Source records → caller-owned model

This one shared authoring cell wires the producer’s complete placed
rings to SCGSim’s public input records. It does not fit geometry, infer
connectivity or choose native object names. The source record’s `::`
keys and qualified occurrence/local Entity IDs are explicitly paired
below. Placed body rings are not transformed again; local neutral
supports carry their one recorded affine.

In [ ]:
if WORKFLOW_ACTION != "analyze":
    source = get_cpw_meander_airbridge_source_records(VARIANT)
    # Geometry classification is separate from physical superconductivity.
    # Preserve the full source record, including is_superconducting.
    geometry_material_kinds = {
        "superconductor": "conductor",
        "conductive": "conductor",
        "conductor": "conductor",
        "dielectric": "dielectric",
        "vacuum": "vacuum",
    }
    entity_ids = {
        e["qualified_entity_id"]: f"{e['source_occurrence_path']}/{e['local_semantic_id']}"
        for e in source["entities"]
    }
    # This notebook explicitly adopts the producer's case grouping proposal.
    # It is not PDK-owned or a connectivity result from the native backend.
    final_nets = {
        entity_ids[source_id]: net
        for group, net in (("signal", "Signal"), ("ground", "Ground"))
        for source_id in source["case_net_groups"][group]
    }
    polygons, entities, conductor_layers, solution_regions = [], [], [], {}
    bounds = dict(
        zip(
            ("x_min_um", "y_min_um", "x_max_um", "y_max_um"),
            source["source_envelope"]["bounds_xy_um"],
            strict=True,
        )
    )
    for e in source["entities"]:
        sid = entity_ids[e["qualified_entity_id"]]
        level = source["source_levels"][e["source_level"]]
        material = source["material_records"][e["source_material"]]
        dielectric = material["material_kind"] == "dielectric"
        low, high = e["source_z_interval_um"]
        geometry = {"z_min_um": low, "z_max_um": high, "thickness_um": level["thickness"]}
        metadata = {
            "source_semantic_id": sid,
            "source_record_id": e["qualified_entity_id"],
            "source_occurrence_path": e["source_occurrence_path"],
            "source_local_entity_id": e["local_semantic_id"],
            "body_membership_id": e["body_membership_id"],
        }
        if "aedt_modeling" in level["info"]:
            metadata["aedt_modeling"] = level["info"]["aedt_modeling"]
        polygon_ids = []
        for p in e["placed_polygons"]:
            polygon_ids.append(p["polygon_id"])
            polygons.append(
                LayoutPolygonSpec(
                    polygon_id=p["polygon_id"],
                    layer="/".join(map(str, p["layer"])),
                    exterior=tuple(map(tuple, p["outer_um"])),
                    holes=tuple(tuple(map(tuple, h)) for h in p["holes_um"]),
                    object_name=sid,
                    net_name=final_nets.get(sid),
                    metadata={"coordinate_frame": p["coordinate_frame"]},
                )
            )
        if len(e["placed_polygons"]) == 1:
            p = e["placed_polygons"][0]
            geometry.update(outer_loop=p["outer_um"], hole_loops=p["holes_um"])
        if dielectric:
            geometry["domain_bounds_um"] = bounds
        record = dict(
            semantic_id=sid,
            role="solution_region" if dielectric else "metal",
            material_id=e["source_material"],
            material_kind=geometry_material_kinds[material["material_kind"]],
            priority=level["mesh_order"],
            geometry_kind="domain" if dielectric else "layout_extrusion",
            geometry=geometry,
            metadata=metadata,
        )
        part_role = None if dielectric else level["info"]["part_role"]
        entities.append(
            SemanticEntitySpec(
                **record,
                polygon_ids=tuple(polygon_ids),
                net_id=final_nets.get(sid),
                part_role=part_role,
                host_void_semantic_id=None if dielectric else "VACUUM_REGION",
            )
        )
        if dielectric:
            solution_regions[sid] = record
        else:
            conductor_layers.append({**record, "part_role": part_role})
    materials = {
        name: {
            **record,
            "kind": geometry_material_kinds[record["material_kind"]],
            "permittivity": record["relative_permittivity"],
        }
        for name, record in source["material_records"].items()
    }
    stack = {
        "materials": materials,
        "layers": conductor_layers,
        "solution_regions": solution_regions,
        "metadata": {},
    }
    source_bottom = min(e["source_z_interval_um"][0] for e in source["entities"])
    source_top = max(e["source_z_interval_um"][1] for e in source["entities"])
    envelope_bottom, envelope_top = source["source_envelope"]["z_interval_um"]
    stack = apply_vacuum_region_to_stack(
        stack,
        VacuumRegionSpec(
            0.0,
            0.0,
            0.0,
            0.0,
            source_bottom - envelope_bottom,
            envelope_top - source_top,
        ),
    )
    vacuum = stack["solution_regions"]["VACUUM_REGION"]
    entities.append(
        SemanticEntitySpec(
            semantic_id="VACUUM_REGION",
            role="solution_region",
            material_id=vacuum["material_id"],
            material_kind="vacuum",
            priority=0,
            geometry_kind="domain",
            geometry=vacuum["geometry"],
            metadata=vacuum["metadata"],
        )
    )
    supports = tuple(LumpedSupport.from_payload(s) for s in source["local_supports"])
    build_input = GeometryBuildInput(
        polygons=tuple(polygons),
        entities=tuple(entities),
        solution_regions=stack["solution_regions"],
        lumped_supports=supports,
        metadata={
            "source_dbu_um": source["source_dbu_um"],
            "caller_final_nets": final_nets,
            "source_coordinate_authority": "placed bodies / local supports",
        },
    )
    display(
        {
            "variant": VARIANT,
            "source_levels": source["source_levels"],
            "source_envelope": source["source_envelope"],
            "entity_count": len(source["entities"]),
            "supports": [s.to_payload() for s in supports],
        }
    )

## Terminal ownership and sweep controls

P1 is left and P2 is right. Each exact Signal Entity and positive Ground
Entity comes from its source declaration. One Ground Entity can own
multiple positive rings; native names/order are not guessed. Impedance
is explicitly 50 Ω, with no renormalization or deembedding. These are
**Driven Terminal Lumped ports**, not Driven Modal ports.

In [ ]:
MODEL = "thin_film"
impedance_ohm = 50.0
resources = AedtResources(cores=4, ram_limit_percent=20)
geometry_workers = 1
run_control = HfssRunControl(
    setup_name="Setup1",
    sweep_name="Sweep1",
    sweep=FrequencySweepSpec(3.0, 8.0, 3),
    maximum_passes=2,
    minimum_passes=1,
    minimum_converged_passes=1,
    maximum_delta_s=0.02,
    percent_refinement=30.0,
)
if WORKFLOW_ACTION != "analyze":
    ports = tuple(
        LumpedTerminalPort(
            index=i,
            name=f"P{i}",
            support_id=s.support_id,
            signal_entity_ids=s.terminal_a_entity_ids,
            reference_entity_ids=s.terminal_b_entity_ids,
            impedance_ohm=impedance_ohm,
            renormalize=False,
            deembed_um=0.0,
        )
        for i, s in enumerate(supports, 1)
    )
    display([p.to_payload() for p in ports])
    display(run_control.to_payload())

The selected source enclosure retains its finite XY and source air top.
HFSS uses the closed default exterior. Numerical controls inherit the
selected short-observation case; neither two passes nor three sweep
samples promises convergence or resolution of the resonator response.

## Prepare / handoff

In [ ]:
HANDOFF = None
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    geometry = prepare_planar_geometry_input(
        build_input,
        prepared_stack=stack,
        modeling=MODEL,
    )
    HANDOFF = prepare_hfss_driven_from_geometry(
        modeling=MODEL,
        geometry=geometry,
        project_name=RUN_ID,
        design_name="MeanderTerminalLumped",
        run_control=run_control,
        ports=ports,
        output_dir=RUN_DIR,
        geometry_workers=geometry_workers,
        resources=resources,
    )
    display(HANDOFF)

Source and effective sheet Z remain distinct. Review the explicit
physical contact association and enabled port settings in the handoff.
Preparation and native execution are not performed in this documentation
round.

## Run: explicit native transaction

In [ ]:
if WORKFLOW_ACTION == "run":
    subprocess.run([str(HANDOFF.script_path)], cwd=HANDOFF.run_dir, check=True)

## Returned Run: fresh kernel / relocated input

In [ ]:
RESULT = None
if WORKFLOW_ACTION == "analyze":
    if ANALYSIS_KIND == "strict":
        RESULT = resolve_results(RETURNED_RUN_DIR)
    elif ANALYSIS_KIND == "receipt_only":
        display(json.loads((RETURNED_RUN_DIR / "metadata/aedt_run_receipt.json").read_text()))
    else:
        raise ValueError("choose strict or receipt_only analysis")

### S-data and resource history

In [ ]:
if RESULT is not None:
    display(RESULT.physics_results())
    display(RESULT.simulation_benchmark())
    display(RESULT.show_all_results(show_details=True))

Inspect actual terminal order/references, S-data frequency and units,
boundary type, pass history and warnings. Keep missing native properties
missing. Port impedance settings do not prove line characteristic
impedance. The [Wave variant](aedt_hfss_driven_terminal_wave.ipynb) is a
different exterior-cut model, not an aligned-plane numerical-equivalence
experiment.